In [3]:
# A typical training procedure for a neural network is as follows

# define nn with some params/weights
# iterate over a dataset of inputs
# process input through the network
# copmute the loss 
# propogate gradients back into the networks params
# update the weights using an update rule


In [21]:
# defining the network
# in this case we're going to create a feed-forward network
import torch
import torch.nn as nn
import torch.nn.functional as F

class PytorchNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 1 input image channel, 6 output channels, 5x5 square convolution
        # kernel
        self.conv1 = nn.Conv2d(1,6,5)
        self.conv2 = nn.Conv2d(6,16,5) # from our 6 output we get those in and output 16 
        # fully connected layers
        self.fc1 = nn.Linear(16 *5*5, 120) 
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84,10) # arbitrary layer params set by LeNet paper

    def forward(self, input):
        # Convolution layer C1: 1 input image channel - 6 output channels
        # 5x5 square convolution using RELU activation function
        # its output a tensor with size (N, 6, 28,28) where N is the size of the batch
        # its shape transform from (N 1, 32 , 32) [which is implied from our architecture and will be explained later]
        # to (N, 6, 28, 28) is because we A - define the second convolution layer to consider 6 feature maps ( an arbitrary number - which we
        # tweak in order to avoid overfitting/minimize compute/storage) and B - the output size of the convolution action
        # is determined by number of valid position the window can take which is formulated as follow
        # output_size = floor((input_size + 2 * padding - kernel_size) / stride) + 1
        # where stride is the amount of overlap we can have between windows and the padding is the number of zeroes we can add to contorl 
        # output size - however since we don't specify those params we use pytorch's defaults - stride = 1, padding = 0 and kernel = 5
        # which simplifies our eq to output_size = input - kernel_size + 1 = 28
        c1 = F.relu(self.conv1(input))
        # iterates in a 2x2 window and preserves largest element from previous layer for example
        # x = torch.tensor([[[[
        #     [1., 2., 3., 4.],
        #     [5., 6., 7., 8.],
        #     [9., 1., 2., 3.],
        #     [4., 5., 6., 7.]
        # ]]]]) 
        # becomes
        # [ 6, 8 ]
        # [ 9, 7 ] 
        s2 = F.max_pool2d(c1,(2,2)) # so outputs a (N, 16, 14,14) tensor
        c3 = F.relu(self.conv2(s2)) # similarly we output (N, 16, [14-5+1 = 10), 10 ) 
        s4 = F.max_pool2d(c3, 2) # cut in half again (N, 16, 5, 5 ) also note: we don't need to specify a tuple this is equivalent to the above
        # then we flattent the batches to feed into the fc 3 layers of the neural net
        s4 = torch.flatten(s4, 1) # (N, 16 * 5 * 5) 
        f5 = F.relu(self.fc1(s4)) # 
        f6 = F.relu(self.fc2(f5))
        return self.fc3(f6) # which outputs a (N, 10) tensor [as implied above]
        # so the reason we can assume the input is because our transforms if we apply  them backward give us those dimension
    

net = PytorchNet()
print(net)
        

PytorchNet(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [27]:
# lets try a random 32x32 input
x = torch.rand(1,1,32,32)
out = net(x)
print(out)

tensor([[-0.0864,  0.0912, -0.0389,  0.0149,  0.0794, -0.0351, -0.0681, -0.0258,
          0.0130, -0.0080]], grad_fn=<AddmmBackward0>)


In [28]:
# as per our routine zero the gradients and seed the params randomly
net.zero_grad()
out.backward(torch.rand(1,10))

In [32]:
# lets define a loss
output = net(x)
target = torch.randn(10)  # a dummy target, for example
target = target.view(1, -1)  # make it the same shape as output
criterion = nn.MSELoss()

loss = criterion(output, target)
print(loss)

tensor(0.8133, grad_fn=<MseLossBackward0>)


In [33]:
# now we backprop
net.zero_grad()     # zeroes the gradient buffers of all parameters

print('conv1.bias.grad before backward')
print(net.conv1.bias.grad)

loss.backward()

print('conv1.bias.grad after backward')
print(net.conv1.bias.grad)

conv1.bias.grad before backward
None
conv1.bias.grad after backward
tensor([ 0.0000, -0.0012, -0.0060,  0.0054, -0.0004,  0.0000])


In [ ]:
# now we follow our update rule - for the sake of this we will use SGD from the pytorch module
import torch.optim as optim
optimizer = optim.SGD(net.parameters(), lr=0.01)
# then in our training loop
optimizer.zero_grad()
output = net(x) # another foward pass
loss = criterion(output, target)
loss.backwards()
optimizer.step() # update the model params